# Exp 2 — Questions Data Pipeline

**Source:** `questions/00_raw.xlsx`

**No dependency on exp_data.** This pipeline does not read anything exp_data produces — it doesn't
need to. Low-accuracy subjects (an exp_data-only criterion — accuracy can't be computed from
question-rating data) are not filtered here at all; they naturally drop out later when exp_data's
Step 10 merge keeps only subjects present in the final exp_data sample. The one thing this pipeline
*does* produce for exp_data to consume is the careless-Likert-responder list (Step 03) — that
exclusion is a whole-subject judgment (not paying attention to the rating questions is a proxy for
not paying attention to the task), so it must be applied to the RT data too, not just the ratings.

Identical structure and logic across all three experiments (except the Exp1 note above).

| Step | File | What happened |
|------|------|---------------|
| 00 | `00_raw.xlsx` | Question rows split from the raw export |
| 01 | `01_cleaned.xlsx` | Filtered to QuestionsList rows, condition + question_type recovered — **no accuracy exclusion** |
| 02 | `02_likert_report.xlsx` | Per-subject dominant-response % and condition-mean range |
| 03 | `03_excl_likert.xlsx` + `03_likert_excluded_subjects.txt` | Careless Likert responders excluded; **excluded-ID list written for exp_data to read directly** |
| 04 | `04_aggregated.xlsx` | Mean rating per subject × action_consistency × question_type — long + wide |

In [1]:
import os
import pandas as pd

base = r'C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\questions'

## Step 00 — Raw question rows

Built externally (E-Prime merge + participant-ID remap) — `00_raw.xlsx` already exists in this
folder; this step just confirms it's there.

In [2]:
df = pd.read_excel(f'{base}\\00_raw.xlsx')
print(f'00_raw.xlsx already exists: {df.shape[0]} rows × {df.shape[1]} cols (built externally, unchanged)')

00_raw.xlsx already exists: 8640 rows × 79 cols (built externally, unchanged)


## Step 01 — Clean: filter to QuestionsList rows, select columns, recover condition + question_type

`Procedure[Trial]` on question rows carries a trailing "1" (e.g. `actionC1`) — marks the
question-block replay of the trial, not a different condition. Stripping it recovers the same
4-level condition used in exp_data.

`QuestionsList` → question_type: 1 = predict (предвидихте), 2 = control (контролирахте),
3 = caused (предизвикахте).

**No accuracy exclusion here** — see header note.

In [3]:
df = pd.read_excel(f'{base}\\00_raw.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

df = df[df['Running[SubTrial]'] == 'QuestionsList'].copy()
print(f'After Running[SubTrial]=QuestionsList: {len(df)} rows')

keep = ['Subject', 'Age', 'Sex', 'Handedness', 'Procedure[Trial]',
        'QuestionsList', 'QuestionsList.Cycle', 'Qs', 'QControl.RESP', 'QControl.RT']
df = df[keep].copy()

df['condition'] = df['Procedure[Trial]'].str.rstrip('1')
print('condition values:', df['condition'].value_counts().to_dict())

df['question_type'] = df['QuestionsList'].map({1.0: 'predict', 2.0: 'control', 3.0: 'caused'})
print('question_type values:', df['question_type'].value_counts().to_dict())
n_subjects = df['Subject'].nunique()
n_rows = len(df)
print(f'Subjects: {n_subjects}  |  Rows: {n_rows}  ({n_subjects} x 144 = {n_subjects*144})')

trial_counts = df.groupby('Subject').size()
bad = trial_counts[trial_counts != 144]
print('Subjects with != 144 rows:', 'none' if bad.empty else bad.to_dict())

out = f'{base}\\01_cleaned.xlsx'
df.to_excel(out, index=False)
print(f'Saved: {out}')

Loaded: 8640 rows × 79 cols
After Running[SubTrial]=QuestionsList: 8640 rows
condition values: {'actionNC': 2880, 'actionC': 2880, 'controlRight': 1440, 'controlLeft': 1440}
question_type values: {'predict': 2880, 'caused': 2880, 'control': 2880}
Subjects: 60  |  Rows: 8640  (60 x 144 = 8640)
Subjects with != 144 rows: none
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\questions\01_cleaned.xlsx


## Step 02 — Likert quality report

dominant-response % (most frequent rating, across all 144 responses) and per-subject condition-mean
range (max − min across the 4-level condition). Same formula across all three experiments.

In [4]:
df = pd.read_excel(f'{base}\\01_cleaned.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

def dominant_pct(s):
    return s.value_counts(normalize=True).iloc[0] * 100

dist = df.groupby('Subject')['QControl.RESP'].apply(dominant_pct).reset_index()
dist.columns = ['Subject', 'dominant_%']

cond_means = df.groupby(['Subject', 'condition'])['QControl.RESP'].mean().unstack()
cond_means['range'] = cond_means.max(axis=1) - cond_means.min(axis=1)
cond_means = cond_means.reset_index()

with pd.ExcelWriter(f'{base}\\02_likert_report.xlsx', engine='openpyxl') as writer:
    dist.to_excel(writer, sheet_name='scale_distribution', index=False)
    cond_means.to_excel(writer, sheet_name='condition_means', index=False)
print(f'Saved: {base}\\02_likert_report.xlsx  (sheets: scale_distribution, condition_means)')

quality = dist.merge(cond_means[['Subject', 'range']], on='Subject')
flagged = quality[(quality['dominant_%'] > 75) & (quality['range'] <= 0.5)].sort_values('Subject')
print(f'Flagged careless (dominant% > 75 AND range <= 0.5): {len(flagged)}')
print(flagged[['Subject', 'dominant_%', 'range']].round(2).to_string(index=False))

kept_flagged = quality[(quality['dominant_%'] > 75) & (quality['range'] > 0.5)].sort_values('Subject')
print(f'Kept despite dominant%>75 flag (range > 0.5): {len(kept_flagged)}')
print(kept_flagged[['Subject', 'dominant_%', 'range']].round(2).to_string(index=False))

Loaded: 8640 rows × 12 cols
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\questions\02_likert_report.xlsx  (sheets: scale_distribution, condition_means)
Flagged careless (dominant% > 75 AND range <= 0.5): 7
 Subject  dominant_%  range
       9       77.78   0.19
      15       98.61   0.17
      22       81.25   0.42
      42       77.08   0.50
      49       86.81   0.48
      65       90.28   0.19
      66       93.75   0.10
Kept despite dominant%>75 flag (range > 0.5): 2
 Subject  dominant_%  range
      41       75.69   1.98
      54       79.86   2.00


## Step 03 — Exclude careless Likert responders + write the excluded-subjects list

This is the one file exp_data reads from this pipeline — `03_likert_excluded_subjects.txt`, a
plain comma-separated list, read dynamically (never hardcoded on the exp_data side).

In [5]:
dist = pd.read_excel(f'{base}\\02_likert_report.xlsx', sheet_name='scale_distribution')
cond = pd.read_excel(f'{base}\\02_likert_report.xlsx', sheet_name='condition_means')
quality = dist[['Subject', 'dominant_%']].merge(cond[['Subject', 'range']], on='Subject')

LIKERT_EXCLUDED = sorted(quality.loc[(quality['dominant_%'] > 75) & (quality['range'] <= 0.5), 'Subject'].tolist())
print(f'Excluded (dominant% > 75 AND range <= 0.5): {LIKERT_EXCLUDED}')

df = pd.read_excel(f'{base}\\01_cleaned.xlsx')
df3 = df[~df['Subject'].isin(LIKERT_EXCLUDED)].copy()
n_subjects = df3['Subject'].nunique()
n_rows = len(df3)
print(f'Remaining: {n_subjects} subjects, {n_rows} rows ({n_subjects} x 144 = {n_subjects*144})')

out = f'{base}\\03_excl_likert.xlsx'
df3.to_excel(out, index=False)
print(f'Saved: {out}')

list_out = f'{base}\\03_likert_excluded_subjects.txt'
with open(list_out, 'w') as f:
    f.write(','.join(str(s) for s in LIKERT_EXCLUDED))
print(f'Excluded-subjects list saved for exp_data pipeline: {list_out}')

Excluded (dominant% > 75 AND range <= 0.5): [9, 15, 22, 42, 49, 65, 66]
Remaining: 53 subjects, 7632 rows (53 x 144 = 7632)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\questions\03_excl_likert.xlsx
Excluded-subjects list saved for exp_data pipeline: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\questions\03_likert_excluded_subjects.txt


## Step 04 — Aggregate mean ratings per subject × action_consistency × question_type

Collapses `controlLeft`/`controlRight` into `control` to get the 3-level `action_consistency` used
throughout. Long format for the 3×3 rANOVA; wide format (with demographics) merged into exp_data's
Step 10.

In [6]:
df = pd.read_excel(f'{base}\\03_excl_likert.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

cond_map = {'actionC': 'consistent', 'actionNC': 'inconsistent',
            'controlLeft': 'control', 'controlRight': 'control'}
df['action_consistency'] = df['condition'].map(cond_map)
print('action_consistency:', df['action_consistency'].value_counts().to_dict())

agg = df.groupby(['Subject', 'action_consistency', 'question_type'])['QControl.RESP'].mean().reset_index()
agg = agg.rename(columns={'Subject': 'sub', 'QControl.RESP': 'rating_mean'})
n_subs = agg['sub'].nunique()
print(f'Long format: {len(agg)} rows ({n_subs} subjects x 9 conditions = {n_subs*9})')

wide = agg.pivot(index='sub', columns=['action_consistency', 'question_type'], values='rating_mean')
wide.columns = [f'{a}_{q}' for a, q in wide.columns]
wide = wide.reset_index()

demo = df.groupby('Subject')[['Age', 'Sex', 'Handedness']].first().reset_index().rename(columns={'Subject': 'sub'})
wide = wide.merge(demo, on='sub', how='left')

print(f'Wide format: {wide.shape[0]} subjects x {wide.shape[1]} columns')
print(list(wide.columns))

out = f'{base}\\04_aggregated.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    agg.to_excel(writer, sheet_name='long', index=False)
    wide.to_excel(writer, sheet_name='wide', index=False)
print(f'Saved: {out}  (sheets: long, wide)')

Loaded: 7632 rows × 12 cols
action_consistency: {'inconsistent': 2544, 'consistent': 2544, 'control': 2544}
Long format: 477 rows (53 subjects x 9 conditions = 477)
Wide format: 53 subjects x 13 columns
['sub', 'consistent_caused', 'consistent_control', 'consistent_predict', 'control_caused', 'control_control', 'control_predict', 'inconsistent_caused', 'inconsistent_control', 'inconsistent_predict', 'Age', 'Sex', 'Handedness']
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\questions\04_aggregated.xlsx  (sheets: long, wide)
